# 02 Cleaning Pipeline\nPurpose: Standardize schema, clean categorical and numeric fields, and export modeling-ready data.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\n\nROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nRAW_PATH = ROOT / 'data' / 'raw' / 'Loan_default.csv'\nCLEAN_OUT = ROOT / 'data' / 'processed' / 'loan_clean.csv'\nLOG_OUT = ROOT / 'data' / 'processed' / '02_cleaning_log.csv'

In [ ]:
df = pd.read_csv(RAW_PATH)\ninitial_rows = len(df)\ninitial_dupes = int(df.duplicated().sum())\ndf = df.drop_duplicates().copy()

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()\nfor c in cat_cols:\n    df[c] = df[c].astype(str).str.strip().str.title()\n\nnum_cols = ['Age','Income','LoanAmount','CreditScore','MonthsEmployed','NumCreditLines','InterestRate','LoanTerm','DTIRatio','Default']\nfor c in num_cols:\n    if c in df.columns:\n        df[c] = pd.to_numeric(df[c], errors='coerce')

In [ ]:
# Conservative missing-value handling for this synthetic-like dataset\nfor c in df.select_dtypes(include=['int64','float64']).columns:\n    df[c] = df[c].fillna(df[c].median())\nfor c in df.select_dtypes(include='object').columns:\n    df[c] = df[c].replace({'Nan': np.nan}).fillna('Unknown')

In [ ]:
df['IncomeToLoanRatio'] = (df['Income'] / df['LoanAmount']).replace([np.inf, -np.inf], np.nan).fillna(0)\ndf['CreditScoreBand'] = pd.cut(df['CreditScore'], bins=[299,579,669,739,799,900], labels=['Poor','Fair','Good','Very Good','Excellent'])\ndf['DTIBand'] = pd.cut(df['DTIRatio'], bins=[0,0.2,0.35,0.5,1.0], labels=['Low','Moderate','High','Critical'], include_lowest=True)

In [ ]:
cleaning_log = pd.DataFrame([\n    {'metric': 'initial_rows', 'value': initial_rows},\n    {'metric': 'rows_after_dedup', 'value': len(df)},\n    {'metric': 'duplicate_rows_removed', 'value': initial_rows - len(df)},\n    {'metric': 'initial_duplicate_rows', 'value': initial_dupes},\n])\nCLEAN_OUT.parent.mkdir(parents=True, exist_ok=True)\ndf.to_csv(CLEAN_OUT, index=False)\ncleaning_log.to_csv(LOG_OUT, index=False)\nprint('saved:', CLEAN_OUT)\nprint('saved:', LOG_OUT)